# 03 - Model Training: TCGA-BRCA

Train and evaluate three classifiers on TCGA pathway-only features (8 features):
- Elastic Net (Logistic Regression with L1/L2)
- Random Forest
- Gradient Boosting

**Setup**: 80/20 stratified train/test split, StandardScaler on features.

**Expected results**: EN AUC=0.793, RF AUC=0.674, GB AUC=0.552

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

from src.data_loader import load_tcga_feature_matrix
from src.models import get_classifiers

## 1. Load Data

In [ ]:
tcga = load_tcga_feature_matrix('../data/processed/02_tcga_feature_matrix.csv')

# Pathway features (8 total)
feature_cols = [c for c in tcga.columns if c.startswith('Pathway_') or c.startswith('Ratio_')]
X = tcga[feature_cols]
y = tcga['high_risk']

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Samples: {len(X)}")
print(f"Class distribution: {dict(y.value_counts())}")

## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Standardize
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train_s.shape[0]} samples")
print(f"Test:  {X_test_s.shape[0]} samples")

## 3. Train and Evaluate Models

In [ ]:
classifiers = get_classifiers()
results = []

for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train_s, y_train)
    
    y_proba = clf.predict_proba(X_test_s)[:, 1]
    y_pred = clf.predict(X_test_s)
    
    auc = roc_auc_score(y_test, y_proba)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({
        'Dataset': 'TCGA',
        'Model': name,
        'AUC': auc,
        'Accuracy': acc,
        'Std': ''
    })
    print(f"  AUC: {auc:.4f}, Accuracy: {acc:.4f}")

results_df = pd.DataFrame(results)
print("\n" + "=" * 50)
print("TCGA RESULTS (Pathway-Only, 8 Features)")
print("=" * 50)
print(results_df.to_string(index=False))

## 4. Save Results

In [ ]:
results_df.to_csv('../results/03_model_performance.csv', index=False)
print("Saved TCGA results to results/03_model_performance.csv")

# Also save elastic net coefficients
en_model = classifiers['Elastic Net']
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': en_model.coef_[0]
}).sort_values('Coefficient', ascending=False)
coef_df.to_csv('../results/04_elastic_net_coefficients.csv', index=False)
print("Saved elastic net coefficients to results/04_elastic_net_coefficients.csv")
print("\n", coef_df.to_string(index=False))